# Sensorless Drive Diagnosis — Fault Classification

**Dataset:** UCI "Dataset for Sensorless Drive Diagnosis" (ID 325)
**Task:** Multi-class classification (11 classes) — diagnose motor drive condition (1 healthy + 10 fault states) from 48 statistical features extracted from electric current signals.
**Models:** Logistic Regression, Decision Tree, K-Nearest Neighbors, Naive Bayes (Gaussian), Random Forest (Ensemble)
**Metrics:** Accuracy, AUC (macro, one-vs-rest), Precision (macro), Recall (macro), F1 (macro), MCC


In [ ]:
import os

In [ ]:
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, roc_auc_score, precision_score,
                              recall_score, f1_score, matthews_corrcoef,
                              confusion_matrix, classification_report)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
sns.set_style("whitegrid")


## 1. Load Dataset

Primary method: UCI's own `ucimlrepo` package (recommended by UCI, resolves the dataset regardless of file-format changes on their end).
Fallback: a local copy at `../data/Sensorless_drive_diagnosis.txt` (whitespace-separated, no header, 48 feature columns + 1 target column) — use this if the lab machine has no internet access or `ucimlrepo` is unavailable.

In [ ]:
from ucimlrepo import fetch_ucirepo
COLUMNS = [f"feature_{i+1}" for i in range(48)] + ["target"]

df = pd.read_csv(
    r"..\data\Sensorless_drive_diagnosis.txt",
    sep=r"\s+",
    header=None,
    names=COLUMNS
)
df = pd.read_csv(
    r"..\data\Sensorless_drive_diagnosis.txt",
    sep=r"\s+",
    header=None,
    names=COLUMNS
)

In [ ]:
N_FEATURES = 48
df.head()


## 2. Exploratory Data Analysis

In [ ]:
df.info()


In [ ]:
df.describe().T.head(10)


In [ ]:
class_counts = df["target"].value_counts().sort_index()
print(class_counts)

plt.figure(figsize=(8, 4))
class_counts.plot(kind="bar", color="#4C72B0")
plt.title("Class distribution — Sensorless Drive Diagnosis")
plt.xlabel("Class (drive condition)")
plt.ylabel("Count")
plt.tight_layout()
plt.show()


The dataset is perfectly balanced across all 11 classes (5,319 rows each), so accuracy is a fair metric here and macro-averaging for precision/recall/F1/AUC won't be skewed by class imbalance.


## 3. Train/Test Split & Preprocessing

In [ ]:
X = df.drop(columns=["target"])
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)
print("Train:", X_train.shape, " Test:", X_test.shape)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


## 4. Train Models & Evaluate

All 5 models


In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(random_state=RANDOM_STATE),
    "kNN": KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
    "Naive Bayes": GaussianNB(),
    "Random Forest (Ensemble)": RandomForestClassifier(
        n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1
    ),
}

results = {}
fitted_models = {}
predictions = {}

for name, model in models.items():
    t0 = time.time()
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)

    results[name] = {
        "Accuracy": accuracy_score(y_test, y_pred),
        "AUC": roc_auc_score(y_test, y_proba, multi_class="ovr", average="macro"),
        "Precision": precision_score(y_test, y_pred, average="macro", zero_division=0),
        "Recall": recall_score(y_test, y_pred, average="macro", zero_division=0),
        "F1": f1_score(y_test, y_pred, average="macro", zero_division=0),
        "MCC": matthews_corrcoef(y_test, y_pred),
    }
    fitted_models[name] = model
    predictions[name] = y_pred
    print(f"{name:28s} trained in {time.time()-t0:5.1f}s  "
          f"acc={results[name]['Accuracy']:.4f}  f1={results[name]['F1']:.4f}  mcc={results[name]['MCC']:.4f}")


## 5. Comparison Table

In [ ]:
results_df = pd.DataFrame(results).T.round(4)
results_df = results_df[["Accuracy", "AUC", "Precision", "Recall", "F1", "MCC"]]
results_df


In [ ]:
results_df.plot(kind="bar", figsize=(11, 5))
plt.title("Model comparison across all metrics")
plt.ylabel("Score")
plt.xticks(rotation=20)
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()


## 6. Confusion Matrix — best-performing model

In [ ]:
best_model_name = results_df["F1"].idxmax()
print("Best model by F1:", best_model_name)

cm = confusion_matrix(y_test, predictions[best_model_name])
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title(f"Confusion Matrix — {best_model_name}")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()

print(classification_report(y_test, predictions[best_model_name]))


## 7. Save Models & Scaler (for the Streamlit app)

In [ ]:
name_to_filename = {
    "Logistic Regression": "logistic_regression.pkl",
    "Decision Tree": "decision_tree.pkl",
    "kNN": "knn.pkl",
    "Naive Bayes": "naive_bayes.pkl",
    "Random Forest (Ensemble)": "random_forest.pkl",
}

for name, model in fitted_models.items():
    joblib.dump(model, name_to_filename[name])
    print("Saved", name_to_filename[name])

joblib.dump(scaler, "scaler.pkl")
results_df.to_csv("model_comparison_results.csv", index=False)
print("Saved scaler.pkl and model_comparison_results.csv")

## 8. Export Sample Test Data

In [ ]:
#  Create Data folder

SAMPLE_SIZE = 1000
PER_CLASS = SAMPLE_SIZE // y_test.nunique()

# Sample indices from each class
sampled_indices = (
    y_test.groupby(y_test)
    .apply(
        lambda x: x.sample(
            n=min(len(x), PER_CLASS),
            random_state=RANDOM_STATE
        )
    )
    .index
)

# Because groupby creates a MultiIndex, get the original indices
sampled_indices = sampled_indices.get_level_values(-1)

# Select corresponding X and y rows
X_sample = X_test.loc[sampled_indices].copy()
y_sample = y_test.loc[sampled_indices].copy()

# Add target column
X_sample["target"] = y_sample.values

test_sample = X_sample.reset_index(drop=True)

# Verify
print("Columns:", test_sample.columns.tolist())
print("Shape:", test_sample.shape)

print("\nClass distribution:")
print(test_sample["target"].value_counts().sort_index())

test_sample.to_csv("../test_data.csv", index=False)
print("\nSaved: test_data.csv")